# Preprocessing
Preprocess data frames from original stat result files.
This notebook is used to look inside the data while preprocessing. 
Actual preprocessing is done in `mininet_vsomeip_dnssec/dataset.py`.

In [296]:
# import sys
# !{sys.executable} -m pip install --quiet polars
# !{sys.executable} -m pip install --quiet matplotlib

In [297]:
# imports and packages
import polars as pl
import os
import ipaddress

# pl.Config.set_tbl_cols(-1)
# pl.Config.set_tbl_rows(-1)

In [ ]:
PROJ_ROOT = ".."
DATA_ROOT = f"{PROJ_ROOT}/data"
RUN_DATE = "20260418-000722"
DATA_RAW = f"{DATA_ROOT}/raw"
DATA_INTERIM = f"{DATA_ROOT}/interim"
DATA_PROCESSED = f"{DATA_ROOT}/processed"
DATA_RAW_CARNET = f"{DATA_RAW}/{RUN_DATE}/carnet"
DATA_RAW_SCALABILITY = f"{DATA_RAW}/{RUN_DATE}/scalability"



## File and name logic and listing raw files
file listings create a nested map of scenario -> series -> config -> run -> service -> file path, e.g. 'carnet': {'H': {'p212_s448': {'17': {'6019': '/home/vm-user/workspace/mininet-vsomeip-evaluation/evaluation/data/raw/20260418-000722/carnet/H-series/run-17/H-6019-#0.csv'}}}}}

In [ ]:
def get_config_for_pubsub_count(pub_count, sub_count):
    return f"p{pub_count}_s{sub_count}"

def get_pubsub_count_from_config(config):
    # Assuming config format is "p{PUBCOUNT}_s{SUBCOUNT}"
    parts = config.split('_')
    pub_count = int(parts[0][1:])  # Extract PUBCOUNT
    sub_count = int(parts[1][1:])  # Extract SUBCOUNT
    return pub_count, sub_count

def find_files_for_series_and_runs_carnet(raw):
    # folder structure in raw dirs is {SERIES}-series/run-{RUN}/files
    # 1. list series folders
    series_folders = [d for d in os.listdir(raw) if (os.path.isdir(os.path.join(raw, d)) and d.endswith('-series'))]
    series_map = {}
    for series in series_folders:
        series_name = series.split('-')[0]  # Extract SERIES
        # No config level in carnet, so use predefined config name based on known pub/sub counts
        config_map = {get_config_for_pubsub_count(212, 448): _find_files_per_run(os.path.join(raw, series))}  
        series_map[series_name] = config_map
    return {"carnet": series_map} ## add a scenario level to be consistent with scalability data structure

def find_files_for_series_and_runs_scalability(raw):
    # folder structure in raw dirs is {scenario}/{SERIES}-series/run-{RUN}/{config}/files
    # 1. list scenario folders
    scenario_folders = [d for d in os.listdir(raw) if os.path.isdir(os.path.join(raw, d))]
    scenario_map = {}
    for scenario in scenario_folders:
        scenario_path = os.path.join(raw, scenario)
        # 2. list series folders
        series_folders = [d for d in os.listdir(scenario_path) if (os.path.isdir(os.path.join(scenario_path, d)) and d.endswith('-series'))]
        series_map = {}
        for series in series_folders:
            series_name = series.split('-')[0]  # Extract SERIES
            config_folders = [d for d in os.listdir(os.path.join(scenario_path, series)) if os.path.isdir(os.path.join(scenario_path, series, d))]
            config_map = {}
            for config in config_folders:
                config_map[config] = _find_files_per_run(os.path.join(scenario_path, series, config))
            series_map[series_name] = config_map
        scenario_map[scenario] = series_map
    return scenario_map

def _find_files_per_run(raw):
    # list files and parse as map run -> map service -> file path
    # 1. list runs in raw/run-{RUN}
    run_folders = [d for d in os.listdir(raw) if (os.path.isdir(os.path.join(raw, d)) and d.startswith('run-'))]
    run_map = {}
    for run in run_folders:
        run_num = run.split('-')[1]  # Extract RUN
        run_map[run_num] = _find_files_per_service(os.path.join(raw, run))
    return run_map

def _find_files_per_service(raw):
    # list files per run and parse as map service -> file path
    # 1. list files in raw dir
    files = [f for f in os.listdir(raw) if (os.path.isfile(os.path.join(raw, f)) and f.endswith('.csv'))]
    # 2. parse file names to extract service name
    service_map = {}
    for f in files:
        # Assuming file name format is "A-SERVICEID-#0.csv"
        service_name = f.split('-')[1]  # Extract SERVICEID
        service_map[service_name] = os.path.join(raw, f)
    # 3. return map service -> file path
    return service_map

In [ ]:
carnet_files = find_files_for_series_and_runs_carnet(DATA_RAW_CARNET)
carnet_files

{'carnet': {'H': {'p212_s448': {'17': {'6019': '../data/raw/20260418-000722/carnet/H-series/run-17/H-6019-#0.csv',
     '2113': '../data/raw/20260418-000722/carnet/H-series/run-17/H-2113-#0.csv',
     '6035': '../data/raw/20260418-000722/carnet/H-series/run-17/H-6035-#0.csv',
     '3007': '../data/raw/20260418-000722/carnet/H-series/run-17/H-3007-#0.csv',
     '7007': '../data/raw/20260418-000722/carnet/H-series/run-17/H-7007-#0.csv',
     '4024': '../data/raw/20260418-000722/carnet/H-series/run-17/H-4024-#0.csv',
     '3042': '../data/raw/20260418-000722/carnet/H-series/run-17/H-3042-#0.csv',
     '6075': '../data/raw/20260418-000722/carnet/H-series/run-17/H-6075-#0.csv',
     '6070': '../data/raw/20260418-000722/carnet/H-series/run-17/H-6070-#0.csv',
     '3010': '../data/raw/20260418-000722/carnet/H-series/run-17/H-3010-#0.csv',
     '6028': '../data/raw/20260418-000722/carnet/H-series/run-17/H-6028-#0.csv',
     '3037': '../data/raw/20260418-000722/carnet/H-series/run-17/H-3037-#0.

In [ ]:
scalability_files = find_files_for_series_and_runs_scalability(DATA_RAW_SCALABILITY)
scalability_files

{'1-50subs_x_1pub': {'H': {'p1_s9': {'17': {'1': '../data/raw/20260418-000722/scalability/1-50subs_x_1pub/H-series/p1_s9/run-17/H-1-#0.csv'},
    '8': {'1': '../data/raw/20260418-000722/scalability/1-50subs_x_1pub/H-series/p1_s9/run-8/H-1-#0.csv'},
    '2': {'1': '../data/raw/20260418-000722/scalability/1-50subs_x_1pub/H-series/p1_s9/run-2/H-1-#0.csv'},
    '5': {'1': '../data/raw/20260418-000722/scalability/1-50subs_x_1pub/H-series/p1_s9/run-5/H-1-#0.csv'},
    '22': {'1': '../data/raw/20260418-000722/scalability/1-50subs_x_1pub/H-series/p1_s9/run-22/H-1-#0.csv'},
    '15': {'1': '../data/raw/20260418-000722/scalability/1-50subs_x_1pub/H-series/p1_s9/run-15/H-1-#0.csv'},
    '13': {'1': '../data/raw/20260418-000722/scalability/1-50subs_x_1pub/H-series/p1_s9/run-13/H-1-#0.csv'},
    '6': {'1': '../data/raw/20260418-000722/scalability/1-50subs_x_1pub/H-series/p1_s9/run-6/H-1-#0.csv'},
    '21': {'1': '../data/raw/20260418-000722/scalability/1-50subs_x_1pub/H-series/p1_s9/run-21/H-1-#0.c

## Result file parsing and preprocessing
Parse result files to a polars data frame and add common aggregates per service, per run, per series.
- Original statistics are capitalized, computed columns are lowercase, `dur` indicates a relative duration between two timestamps, see `column_duration_pairs`.  
- Some original columns are dropped as they are not used, see `columns_to_drop`.
- All timestamps and durations are in nanoseconds.

Goal is to generate one data frame per config that contains aggregated statistics over all runs of that config and can be used for further analysis.

In [302]:
# Helpers to parse one stat file
columns_to_drop = [
    "PUBLISHER_APP_INITIALIZATION",
    "GENERATE_OFFER_NONCE_START",
    "GENERATE_OFFER_NONCE_END",
    "FIND_RECEIVE",
    "OFFER_SEND",
    "SVCB_SERVICE_REQUEST_RECEIVE",
    "SVCB_SERVICE_RESPONSE_SEND",
    "SVCB_CLIENT_REQUEST_SEND",
    "SVCB_CLIENT_REQUEST_RECEIVE",
    "SVCB_CLIENT_RESPONSE_SEND",
    "SVCB_CLIENT_RESPONSE_RECEIVE",
    "TLSA_CLIENT_REQUEST_RECEIVE",
    "TLSA_CLIENT_RESPONSE_SEND",
    "TLSA_SERVICE_REQUEST_RECEIVE",
    "TLSA_SERVICE_RESPONSE_SEND",
]

column_duration_pairs = {
    ("SUBSCRIBER_APP_INITIALIZATION_START", "SUBSCRIBER_APP_INITIALIZATION_END"): "subscriber_app_init_dur",
    ("FIND_SEND", "OFFER_RECEIVE"): "find_offer_dur",
    ("SVCB_SERVICE_REQUEST_SEND", "SVCB_SERVICE_RESPONSE_RECEIVE"): "svcb_service_dur",
    ("VALIDATE_OFFER_START", "VALIDATE_OFFER_END"): "validate_offer_dur",
    ("CLIENT_SIGN_START", "CLIENT_SIGN_END"): "client_sign_dur",
    ("OFFER_RECEIVE", "SUBSCRIBE_SEND"): "offer_to_subscribe_dur",
    ("SUBSCRIBE_SEND", "SUBSCRIBE_RECEIVE"): "subscribe_transmission_dur",
    ("TLSA_CLIENT_REQUEST_SEND", "TLSA_CLIENT_RESPONSE_RECEIVE"): "tlsa_client_dur",
    ("VERIFY_CLIENT_SIGNATURE_START", "VERIFY_CLIENT_SIGNATURE_END"): "verify_client_dur",
    ("SERVICE_SIGN_START", "SERVICE_SIGN_END"): "service_sign_dur",
    ("SUBSCRIBE_ACK_SEND", "SUBSCRIBE_ACK_RECEIVE"): "subscribeack_transmission_dur",
    ("SUBSCRIBE_SEND", "SUBSCRIBE_ACK_RECEIVE"): "subscribe_to_subscribeack_client_dur",
    ("SUBSCRIBE_RECEIVE", "SUBSCRIBE_ACK_SEND"): "subscribe_to_subscribeack_service_dur",
    ("TLSA_SERVICE_REQUEST_SEND", "TLSA_SERVICE_RESPONSE_RECEIVE"): "tlsa_service_dur",
    ("VERIFY_SERVICE_SIGNATURE_START", "VERIFY_SERVICE_SIGNATURE_END"): "verify_service_dur",
    ("OFFER_RECEIVE", "SUBSCRIBE_ACK_RECEIVE"): "offer_receive_to_subscribeack_dur",
    ("VALIDATE_OFFER_END", "SUBSCRIBE_ACK_RECEIVE"): "validate_offer_to_subscribeack_dur",
    ("OFFER_RECEIVE", "VERIFY_SERVICE_SIGNATURE_END"): "offer_receive_to_verify_service_dur",
    ("VALIDATE_OFFER_END", "VERIFY_SERVICE_SIGNATURE_END"): "validate_offer_to_verify_service_dur",
}

def preprocess_stat_file(raw_file):
    duration_exprs = [
        (pl.col(end_col) - pl.col(start_col)).alias(duration_col)
        for (start_col, end_col), duration_col in column_duration_pairs.items()
    ]
    # host_ip_expr = pl.col("HOST").map_elements(lambda x: str(ipaddress.IPv4Address(x)), return_dtype=pl.Utf8).alias("host_ip")
    return (
        pl.scan_csv(raw_file)
        .drop(columns_to_drop, strict=False)
        .with_columns(pl.selectors.numeric().replace(0, None)) # replace 0 with null for numeric columns to avoid skewing aggregates
        .with_columns(duration_exprs)
        .with_columns(
            [
                pl.max_horizontal(
                    "offer_receive_to_subscribeack_dur",
                    "offer_receive_to_verify_service_dur",
                ).alias("subscription_dur"),
                pl.max_horizontal(
                    "validate_offer_to_subscribeack_dur",
                    "validate_offer_to_verify_service_dur",
                ).alias("valid_offer_to_subscription_dur"),
                pl.min_horizontal(
                    "svcb_service_dur",
                    "tlsa_client_dur",
                    "tlsa_service_dur",
                ).alias("dns_resolution_min_dur"),
                pl.mean_horizontal(
                    "svcb_service_dur",
                    "tlsa_client_dur",
                    "tlsa_service_dur",
                ).alias("dns_resolution_mean_dur"),
                pl.max_horizontal(
                    "svcb_service_dur",
                    "tlsa_client_dur",
                    "tlsa_service_dur",
                ).alias("dns_resolution_max_dur"),
            ]
        )
        .collect()
    )



In [303]:
# helpers to parse one run with multiple service files
cols_duration = list(column_duration_pairs.values()) + [
    "subscription_dur",
    "valid_offer_to_subscription_dur",
]
cols_first_last = [
        ("SUBSCRIBER_APP_INITIALIZATION_START", "subscriber_app_initialization_start"),
        ("OFFER_RECEIVE", "offer_receive"),
        ("VALIDATE_OFFER_END", "validate_offer_end"),
        ("FIND_SEND", "find_send"),
        ("SUBSCRIBE_SEND", "subscribe_send"),
        ("SUBSCRIBE_ACK_RECEIVE", "subscribe_ack_receive"),
        ("VERIFY_SERVICE_SIGNATURE_END", "verify_service_signature_end"),
        ("SVCB_SERVICE_REQUEST_SEND", "svcb_service_request_send"),
        ("TLSA_CLIENT_REQUEST_SEND", "tlsa_client_request_send"),
        ("TLSA_SERVICE_REQUEST_SEND", "tlsa_service_request_send"),
        ("SUBSCRIBER_APP_INITIALIZATION_END", "subscriber_app_initialization_end"),
    ]

def preprocess_run_data(raw_run_files):
    # preprocess and stack all service files for one run
    all_services_df = pl.concat(
        [preprocess_stat_file(path).with_columns(pl.lit(service).alias("service")) for service, path in raw_run_files.items()],
        how="vertical_relaxed",
        rechunk=True,
    )
    # build aggregate expressions once
    first_last_exprs = [
        expr
        for col_name, alias_base in cols_first_last
        for expr in (
            pl.col(col_name).drop_nulls().min().alias(f"{alias_base}_first"),
            pl.col(col_name).drop_nulls().max().alias(f"{alias_base}_last"),
        )
    ]
    duration_exprs = [
        expr
        for col_name in cols_duration
        for expr in (
            pl.col(col_name).drop_nulls().min().alias(f"{col_name}_min"),
            pl.col(col_name).drop_nulls().mean().alias(f"{col_name}_mean"),
            pl.col(col_name).drop_nulls().std().alias(f"{col_name}_stddev"),
            pl.col(col_name).drop_nulls().max().alias(f"{col_name}_max"),
        )
    ]
    dns_exprs = [
        pl.col("dns_resolution_min_dur").drop_nulls().min().alias("dns_resolution_dur_min"),
        pl.col("dns_resolution_mean_dur").drop_nulls().mean().alias("dns_resolution_dur_mean"),
        pl.col("dns_resolution_max_dur").drop_nulls().max().alias("dns_resolution_dur_max"),
    ]
    # expression to drop everything except the columns that contain "dur"
    drop_non_duration_expr = pl.all().exclude("^.*dur.*$")
    # compute all base aggregates in one pass, then derive total durations
    return (
        all_services_df
        .select(first_last_exprs + duration_exprs + dns_exprs)
        .with_columns(
            (pl.col("subscribe_ack_receive_last") - pl.col("offer_receive_first")).alias(
                "total_offer_receive_to_subscribeack_dur"
            ),
            (pl.col("verify_service_signature_end_last") - pl.col("offer_receive_first")).alias(
                "total_offer_receive_to_verify_service_dur"
            ),
            (pl.col("subscribe_ack_receive_last") - pl.col("validate_offer_end_first")).alias(
                "total_validate_offer_to_subscribeack_dur"
            ),
            (pl.col("verify_service_signature_end_last") - pl.col("validate_offer_end_first")).alias(
                "total_validate_offer_to_verify_service_dur"
            ),
        ).drop(drop_non_duration_expr)
    ), all_services_df

In [ ]:
# helpers to parse all runs of one config 
def preprocess_config_runs(raw_config_runs):
    # preprocess and stack all runs for one config, including run number
    all_runs_df = pl.concat(
        [preprocess_run_data(data)[0].with_columns(pl.lit(runs).cast(pl.Int32).alias("run")) for runs, data in raw_config_runs.items()],
        how="vertical_relaxed",
        rechunk=True,
    )
    # create another dataframe that contains the aggregates of these runs
    
    return all_runs_df

In [ ]:
# example for file processing of one service file
df_stat = preprocess_stat_file(carnet_files['carnet']['H']['p212_s448']['18']['2114'])
df_stat

HOST,SUBSCRIBER_APP_INITIALIZATION_START,SUBSCRIBER_APP_INITIALIZATION_END,FIND_SEND,OFFER_RECEIVE,SVCB_SERVICE_REQUEST_SEND,SVCB_SERVICE_RESPONSE_RECEIVE,VALIDATE_OFFER_START,VALIDATE_OFFER_END,CLIENT_SIGN_START,CLIENT_SIGN_END,SUBSCRIBE_SEND,SUBSCRIBE_RECEIVE,TLSA_CLIENT_REQUEST_SEND,TLSA_CLIENT_RESPONSE_RECEIVE,VERIFY_CLIENT_SIGNATURE_START,VERIFY_CLIENT_SIGNATURE_END,SERVICE_SIGN_START,SERVICE_SIGN_END,SUBSCRIBE_ACK_SEND,SUBSCRIBE_ACK_RECEIVE,TLSA_SERVICE_REQUEST_SEND,TLSA_SERVICE_RESPONSE_RECEIVE,VERIFY_SERVICE_SIGNATURE_START,VERIFY_SERVICE_SIGNATURE_END,subscriber_app_init_dur,find_offer_dur,svcb_service_dur,validate_offer_dur,client_sign_dur,offer_to_subscribe_dur,subscribe_transmission_dur,tlsa_client_dur,verify_client_dur,service_sign_dur,subscribeack_transmission_dur,subscribe_to_subscribeack_client_dur,subscribe_to_subscribeack_service_dur,tlsa_service_dur,verify_service_dur,offer_receive_to_subscribeack_dur,validate_offer_to_subscribeack_dur,offer_receive_to_verify_service_dur,validate_offer_to_verify_service_dur,subscription_dur,valid_offer_to_subscription_dur,dns_resolution_min_dur,dns_resolution_mean_dur,dns_resolution_max_dur
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,i64
167772161,1776464198759350248,1776464198760877419,null,null,1776464198791755781,1776464198793372010,1776464198793382317,1776464198793382636,1776464198796788775,1776464198798490169,1776464198798495624,1776464198798614770,1776464198798627132,1776464198799014514,1776464198800330084,1776464198800623130,1776464198801808001,1776464198803396661,1776464198803401481,1776464198803879512,1776464198793457688,1776464198793947702,1776464198803889530,1776464198804133145,1527171,null,1616229,319,1701394,null,119146,387382,293046,1588660,478031,5383888,4786711,490014,243615,null,10496876,null,10750509,null,10750509,387382,831208.333333,1616229


In [ ]:
# example for processing one run with all its service files
df_run, all_services_df = preprocess_run_data(carnet_files['carnet']['H']['p212_s448']['18'])
# all_services_df
df_run

subscriber_app_init_dur_min,subscriber_app_init_dur_mean,subscriber_app_init_dur_stddev,subscriber_app_init_dur_max,find_offer_dur_min,find_offer_dur_mean,find_offer_dur_stddev,find_offer_dur_max,svcb_service_dur_min,svcb_service_dur_mean,svcb_service_dur_stddev,svcb_service_dur_max,validate_offer_dur_min,validate_offer_dur_mean,validate_offer_dur_stddev,validate_offer_dur_max,client_sign_dur_min,client_sign_dur_mean,client_sign_dur_stddev,client_sign_dur_max,offer_to_subscribe_dur_min,offer_to_subscribe_dur_mean,offer_to_subscribe_dur_stddev,offer_to_subscribe_dur_max,subscribe_transmission_dur_min,subscribe_transmission_dur_mean,subscribe_transmission_dur_stddev,subscribe_transmission_dur_max,tlsa_client_dur_min,tlsa_client_dur_mean,tlsa_client_dur_stddev,tlsa_client_dur_max,verify_client_dur_min,verify_client_dur_mean,verify_client_dur_stddev,verify_client_dur_max,service_sign_dur_min,service_sign_dur_mean,service_sign_dur_stddev,service_sign_dur_max,subscribeack_transmission_dur_min,subscribeack_transmission_dur_mean,subscribeack_transmission_dur_stddev,subscribeack_transmission_dur_max,subscribe_to_subscribeack_client_dur_min,subscribe_to_subscribeack_client_dur_mean,subscribe_to_subscribeack_client_dur_stddev,subscribe_to_subscribeack_client_dur_max,subscribe_to_subscribeack_service_dur_min,subscribe_to_subscribeack_service_dur_mean,subscribe_to_subscribeack_service_dur_stddev,subscribe_to_subscribeack_service_dur_max,tlsa_service_dur_min,tlsa_service_dur_mean,tlsa_service_dur_stddev,tlsa_service_dur_max,verify_service_dur_min,verify_service_dur_mean,verify_service_dur_stddev,verify_service_dur_max,offer_receive_to_subscribeack_dur_min,offer_receive_to_subscribeack_dur_mean,offer_receive_to_subscribeack_dur_stddev,offer_receive_to_subscribeack_dur_max,validate_offer_to_subscribeack_dur_min,validate_offer_to_subscribeack_dur_mean,validate_offer_to_subscribeack_dur_stddev,validate_offer_to_subscribeack_dur_max,offer_receive_to_verify_service_dur_min,offer_receive_to_verify_service_dur_mean,offer_receive_to_verify_service_dur_stddev,offer_receive_to_verify_service_dur_max,validate_offer_to_verify_service_dur_min,validate_offer_to_verify_service_dur_mean,validate_offer_to_verify_service_dur_stddev,validate_offer_to_verify_service_dur_max,subscription_dur_min,subscription_dur_mean,subscription_dur_stddev,subscription_dur_max,valid_offer_to_subscription_dur_min,valid_offer_to_subscription_dur_mean,valid_offer_to_subscription_dur_stddev,valid_offer_to_subscription_dur_max,dns_resolution_dur_min,dns_resolution_dur_mean,dns_resolution_dur_max,total_offer_receive_to_subscribeack_dur,total_offer_receive_to_verify_service_dur,total_validate_offer_to_subscribeack_dur,total_validate_offer_to_verify_service_dur
i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,i64,i64,i64,i64,i64
1524964,1.0596e7,3.4994e6,13425695,259405,4.4183e8,2.4332e8,1053787976,96174,3.4774e6,6.0642e6,51117066,64,145.174107,80.5216,622,521430,789637.267857,1.8572e6,38435290,577886,1.7324e7,1.2525e7,117873819,61638,3.1094e7,2.8033e7,95341541,127385,7.1218e6,8.0299e6,41066122,87319,119432.091518,44183.585522,304748,525110,725090.642857,517283.60342,4495615,48680,1.5058e7,2.1385e7,131464786,1494325,5.8781e7,3.9014e7,318360181,1344093,1.2630e7,1.6906e7,265047809,62303,3.1670e6,4.4873e6,23177349,80355,112747.316964,54265.926567,808027,3855159,7.6225e7,4.4144e7,328103459,3851129,7.6061e7,4.4218e7,328094649,3971440,7.6344e7,4.4140e7,328222528,3967410,7.6181e7,4.4214e7,328213718,3971440,7.6344e7,4.4140e7,328222528,3967410,7.6181e7,4.4214e7,328213718,62303,4.5887e6,51117066,1288189887,1288304029,1288179666,1288293808


In [ ]:
# example for processing all runs of one config
df_config = preprocess_config_runs(carnet_files['carnet']['H']['p212_s448'])
df_config

subscriber_app_init_dur_min,subscriber_app_init_dur_mean,subscriber_app_init_dur_stddev,subscriber_app_init_dur_max,find_offer_dur_min,find_offer_dur_mean,find_offer_dur_stddev,find_offer_dur_max,svcb_service_dur_min,svcb_service_dur_mean,svcb_service_dur_stddev,svcb_service_dur_max,validate_offer_dur_min,validate_offer_dur_mean,validate_offer_dur_stddev,validate_offer_dur_max,client_sign_dur_min,client_sign_dur_mean,client_sign_dur_stddev,client_sign_dur_max,offer_to_subscribe_dur_min,offer_to_subscribe_dur_mean,offer_to_subscribe_dur_stddev,offer_to_subscribe_dur_max,subscribe_transmission_dur_min,subscribe_transmission_dur_mean,subscribe_transmission_dur_stddev,subscribe_transmission_dur_max,tlsa_client_dur_min,tlsa_client_dur_mean,tlsa_client_dur_stddev,tlsa_client_dur_max,verify_client_dur_min,verify_client_dur_mean,verify_client_dur_stddev,verify_client_dur_max,service_sign_dur_min,service_sign_dur_mean,service_sign_dur_stddev,service_sign_dur_max,subscribeack_transmission_dur_min,subscribeack_transmission_dur_mean,subscribeack_transmission_dur_stddev,subscribeack_transmission_dur_max,subscribe_to_subscribeack_client_dur_min,subscribe_to_subscribeack_client_dur_mean,subscribe_to_subscribeack_client_dur_stddev,subscribe_to_subscribeack_client_dur_max,subscribe_to_subscribeack_service_dur_min,subscribe_to_subscribeack_service_dur_mean,subscribe_to_subscribeack_service_dur_stddev,subscribe_to_subscribeack_service_dur_max,tlsa_service_dur_min,tlsa_service_dur_mean,tlsa_service_dur_stddev,tlsa_service_dur_max,verify_service_dur_min,verify_service_dur_mean,verify_service_dur_stddev,verify_service_dur_max,offer_receive_to_subscribeack_dur_min,offer_receive_to_subscribeack_dur_mean,offer_receive_to_subscribeack_dur_stddev,offer_receive_to_subscribeack_dur_max,validate_offer_to_subscribeack_dur_min,validate_offer_to_subscribeack_dur_mean,validate_offer_to_subscribeack_dur_stddev,validate_offer_to_subscribeack_dur_max,offer_receive_to_verify_service_dur_min,offer_receive_to_verify_service_dur_mean,offer_receive_to_verify_service_dur_stddev,offer_receive_to_verify_service_dur_max,validate_offer_to_verify_service_dur_min,validate_offer_to_verify_service_dur_mean,validate_offer_to_verify_service_dur_stddev,validate_offer_to_verify_service_dur_max,subscription_dur_min,subscription_dur_mean,subscription_dur_stddev,subscription_dur_max,valid_offer_to_subscription_dur_min,valid_offer_to_subscription_dur_mean,valid_offer_to_subscription_dur_stddev,valid_offer_to_subscription_dur_max,dns_resolution_dur_min,dns_resolution_dur_mean,dns_resolution_dur_max,total_offer_receive_to_subscribeack_dur,total_offer_receive_to_verify_service_dur,total_validate_offer_to_subscribeack_dur,total_validate_offer_to_verify_service_dur,run
i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,f64,i64,i64,f64,i64,i64,i64,i64,i64,i32
1405929,1.0548e7,3.4895e6,13037139,204930,3.3209e8,2.5621e8,1041655572,67331,4.0249e6,6.6267e6,51979990,59,7041.930804,146074.438734,3091953,523351,666548.5625,491774.652142,5593006,602944,1.8765e7,1.1805e7,50340524,85195,4.7095e7,3.6369e7,108014645,87110,1.6561e7,1.6493e7,51087214,87457,136647.890625,219343.739268,3357296,528059,742901.294643,568879.708002,5901870,48314,2.2365e7,2.6524e7,93041074,2273137,9.1699e7,5.3768e7,420912987,1385292,2.2239e7,2.3926e7,352479127,45391,3.5106e6,5.7051e6,36146642,80279,144634.078125,590189.797225,12427672,6064791,1.1046e8,6.2394e7,444064923,6055129,1.1045e8,6.2397e7,444062337,6633373,1.1062e8,6.2361e7,444176353,6623711,1.1060e8,6.2364e7,444173767,6633373,1.1062e8,6.2361e7,444176353,6623711,1.1060e8,6.2364e7,444173767,45391,8.0322e6,51979990,1245225168,1245318091,1245215506,1245308429,17
1408769,1.2346e7,4.6412e6,16681451,24752

In [ ]:
# export df_config to parquet for use in analysis notebook 